In [ ]:
import os
import subprocess
from pathlib import Path
import gdown
import zipfile
import yaml
import re
import json

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt

# 1. Клонирование репозитория

In [ ]:
REPO_URL = "https://github.com/Lzora7/Study.git" 
SUBFOLDER = "AI-HSE/Sound/HW_2_dpspch"
REPO_NAME = 'Study'

target_dir = Path(REPO_NAME) / SUBFOLDER

# не клонирован ли уже репозиторий
if target_dir.exists():
    print(f"Репозиторий уже находится в: {target_dir}")
    os.chdir(target_dir)
    print(f"Перешли в директорию: {Path.cwd()}")
else:
    print(f"Клонирование репозитория из: {REPO_URL}")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
        repo_dir = Path(REPO_NAME)
        if not repo_dir.exists():
            print(f"Ошибка: репозиторий не был клонирован в {repo_dir}")
        elif SUBFOLDER:
            # переходим в подпапку
            os.chdir(target_dir)
            print(f"Репозиторий клонирован, перешли в: {Path.cwd()}")
    else:
        print(f"Ошибка при клонировании: {result.stderr}")

In [ ]:
# зависимости
%pip install -q -r requirements.txt

import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Установка завершена")

## 2. Загрузка весов модели

Для работы inference нужны предобученные веса модели. Они должны быть сохранены в директории `saved/`.

**Формат:**
- Модель: `saved/{run_name}/model_best.pth`
- Конфигурация: `saved/{run_name}/config.yaml`

Вставьте ссылку на Google Drive с весами модели в следующую ячейку.

In [ ]:
print("ШАГ 2: ЗАГРУЗКА ВЕСОВ МОДЕЛИ")

# ссылка на веса из google drive
MODEL_WEIGHTS = "https://drive.google.com/file/d/1tRMQY3NXqhuPEXXs9NG33Q3TKCVy_tMz/view?usp=sharing" 

# преобразуем ссылку в правильный формат для gdown
def convert_google_drive_link(link):
    """
    Преобразует ссылку Google Drive формата 
    https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    в формат для gdown: https://drive.google.com/uc?id=FILE_ID
    """
    
    if "drive.google.com/uc?id=" in link:
        return link
    
    # извлекаем FILE_ID из формата https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    pattern = r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)'
    match = re.search(pattern, link)
    
    if match:
        file_id = match.group(1)
        # преобразуем в формат для gdown
        converted_link = f"https://drive.google.com/uc?id={file_id}"
        print(f"Ссылка преобразована: {converted_link}")
        return converted_link
    else:
        print(f"Не удалось извлечь FILE_ID из ссылки.")
        print(f"Ожидается формат: https://drive.google.com/file/d/FILE_ID/view?usp=sharing")
        return link

# Преобразуем ссылку
MODEL_WEIGHTS = convert_google_drive_link(MODEL_WEIGHTS)
print(f"Загрузка модели из: {MODEL_WEIGHTS}")
    
# директория для сохранения
saved_dir = Path("saved")
saved_dir.mkdir(exist_ok=True)

# скачивание с использованием правильного формата
output_file = saved_dir / "model_best.pth"

# удаляем файл для перезаписи
if output_file.exists():
    output_file.unlink()
    print(f"Удален старый файл: {output_file}")

# скачиваем файл
print(f"Скачивание файла...")
try:
    gdown.download(MODEL_WEIGHTS, str(output_file), quiet=False, fuzzy=True)
except Exception as e:
    print(f"Ошибка при скачивании: {e}")

## 3. Запуск inference.py

Скрипт `inference.py` позволяет:
- Загрузить предобученную модель из checkpoint
- Выполнить распознавание речи на тестовых данных
- Вычислить метрики WER/CER
- Сохранить предсказания модели в файлы

### Базовое использование:

```bash
python inference.py inferencer.from_pretrained=saved/<model_name>/model_best.pth
```

### Использование с кастомным датасетом:

```bash
python inference.py \
    datasets=custom_dataset \
    datasets.custom_dataset.test.audio_dir=/path/to/audio \
    datasets.custom_dataset.test.transcription_dir=/path/to/transcriptions \
    inferencer.from_pretrained=saved/<model_name>/model_best.pth \
    inferencer.save_path=custom_dataset_results
```

### Параметры:

- `inferencer.from_pretrained` - путь к checkpoint модели
- `datasets` - название конфигурации датасета (например, `custom_dataset`)
- `datasets.custom_dataset.test.audio_dir` - путь к папке с аудио файлами
- `datasets.custom_dataset.test.transcription_dir` - путь к папке с транскрипциями (опционально)
- `inferencer.save_path` - имя папки для сохранения результатов

Результаты сохраняются в `data/saved/<save_path>/test/` в виде файлов `{UtteranceID}.txt` с предсказаниями.

**Примечание:** Практический пример запуска inference на ваших данных показан в следующем разделе.

## 4. Работа с собственными данными из Google Drive

Для тестирования на вашем датасете:

1. Подготовьте датасет в следующем формате:
   ```
   NameOfTheDirectoryWithUtterances
   ├── audio
   │   ├── UtteranceID1.wav  # может быть .flac или .mp3
   │   ├── UtteranceID2.wav
   │   └── ...
   └── transcriptions  # опционально - ground truth
       ├── UtteranceID1.txt
       ├── UtteranceID2.txt
       └── ...
   ```

2. Загрузите датасет на Google Drive

3. Вставьте ссылку на Google Drive в следующую ячейку

4. Запустите inference для получения предсказаний

5. Если есть транскрипции, используйте calc_metrics.py для подсчета WER/CER

In [ ]:
print("ЗАГРУЗКА ДАТАСЕТА ИЗ GOOGLE DRIVE")

# Вставьте ссылку на Google Drive с датасетом
# * сейчас тут ссылка на подобие такого датасета сделанного из семпла train-clean-100
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1Z1gyJZcUAaw7STT2vYcSMC4X97NtXILt/view?usp=sharing" 

def convert_google_drive_link(link):
    """
    Преобразует ссылку Google Drive формата 
    https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    в формат для gdown: https://drive.google.com/uc?id=FILE_ID
    """
    
    # если правильный формат для gdown, возвращаем как есть
    if "drive.google.com/uc?id=" in link:
        return link
    
    # Извлекаем FILE_ID
    pattern = r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)'
    match = re.search(pattern, link)
    
    if match:
        file_id = match.group(1)
        # преобразуем в формат для gdown
        converted_link = f"https://drive.google.com/uc?id={file_id}"
        print(f"Ссылка преобразована: {converted_link}")
        return converted_link
    else:
        print(f"Не удалось извлечь FILE_ID из ссылки.")
        print(f"Ожидается формат: https://drive.google.com/file/d/FILE_ID/view?usp=sharing")
        return link

if DATASET_DRIVE_LINK:
    # преобразуем ссылку в правильный формат
    DATASET_DRIVE_LINK = convert_google_drive_link(DATASET_DRIVE_LINK)
    print(f"Загрузка датасета из: {DATASET_DRIVE_LINK}")
    
    # создаем директорию для датасета
    custom_dataset_dir = Path("data/custom_dataset")
    custom_dataset_dir.mkdir(parents=True, exist_ok=True)
    
    # скачиваем файл
    output_file = custom_dataset_dir / "dataset.zip"
    gdown.download(DATASET_DRIVE_LINK, str(output_file), quiet=False)
    
    # проверяем, что получилось
    if not output_file.exists():
        print("Файл не скачался")
    else:
        file_size = output_file.stat().st_size
        file_size_mb = file_size / 1024 / 1024
        print(f"Файл загружен: {output_file}")

        # проверяем, является ли файл zip-архивом
        is_zip = False
        if output_file.suffix.lower() == ".zip":
            is_zip = True
        
        # распаковываем, если zip
        if is_zip:
            try:
                print(f"Распаковка архива...")
                with zipfile.ZipFile(output_file, 'r') as zip_ref:
                    zip_ref.extractall(custom_dataset_dir)
                output_file.unlink() # удаляем zip файл
                print(f"Архив распакован")
            except Exception as e:
                print(f"Ошибка при распаковке: {e}")
        else:
            print(f"Файл не является zip архивом (расширение: {output_file.suffix})")
            print(f"Пропускаем распаковку. Убедитесь, что датасет уже распакован.")
    
    print(f"Датасет загружен в: {custom_dataset_dir}")
    
    # проверка наличия аудио
    audio_files = list(custom_dataset_dir.glob("**/*.wav")) + \
                  list(custom_dataset_dir.glob("**/*.flac")) + \
                  list(custom_dataset_dir.glob("**/*.mp3"))
    
    print(f"\nНайдено аудио файлов: {len(audio_files)}")
    
    if len(audio_files) == 0:
        print("Аудио файлы не найдены")
        print("Убедитесь, что датасет содержит файлы .wav, .flac или .mp3")
else:
    print("Ссылка на Google Drive не указана.")

In [ ]:
print("СОЗДАНИЕ КОНФИГУРАЦИИ И ЗАПУСК INFERENCE")

# путь к кастомному датасету
custom_dataset_dir = Path("data/custom_dataset")

# ищем папки с аудио и транскрипциями
audio_dir = None
transcription_dir = None

# проверяем структуру датасета
if custom_dataset_dir.exists():

    # ищем папки audio и transcriptions
    for subdir in custom_dataset_dir.rglob("*"):
        if subdir.is_dir():
            dir_name_lower = subdir.name.lower()
            if "audio" in dir_name_lower and audio_dir is None:
                # проверка, что в папке есть аудио файлы
                audio_files_in_dir = list(subdir.glob("*.wav")) + \
                                     list(subdir.glob("*.flac")) + \
                                     list(subdir.glob("*.mp3"))
                if audio_files_in_dir:
                    audio_dir = subdir
            elif ("transcript" in dir_name_lower or "text" in dir_name_lower) and transcription_dir is None:
                # проверка, что в папке есть txt файлы
                txt_files_in_dir = list(subdir.glob("*.txt"))
                if txt_files_in_dir:
                    transcription_dir = subdir

if audio_dir:
    print(f"Найдена папка с аудио: {audio_dir}")
    if transcription_dir:
        print(f"Найдена папка с транскрипциями: {transcription_dir}")
    else:
        print(f"Папка с транскрипциями не найдена (опциональная)")
    
    # создаем конфигурацию для кастомного датасета
    config_dir = Path("src/configs/datasets")
    config_file = config_dir / "custom_dataset.yaml"
    
    config_data = {
        "test": {
            "_target_": "src.datasets.CustomDirAudioDataset",
            "audio_dir": str(audio_dir.absolute()),
            "transcription_dir": str(transcription_dir.absolute()) if transcription_dir else None,
            "instance_transforms": "${transforms.instance_transforms.inference}"
        }
    }
    
    with open(config_file, 'w') as f:
        yaml.dump(config_data, f, default_flow_style=False)
    
    print(f"\nКонфигурация создана: {config_file}")
    
    # сохраняем информацию о найденных папках для следующего шага
    dataset_info = {
        "audio_dir": str(audio_dir.absolute()),
        "transcription_dir": str(transcription_dir.absolute()) if transcription_dir else None
    }
    info_file = custom_dataset_dir / "dataset_info.json"
    with open(info_file, 'w') as f:
        json.dump(dataset_info, f, indent=2)
    print(f"Информация о датасете сохранена: {info_file}")
    
    # Устанавливаем batch_size для inference = 1, так как базовый он = 32
    batch_size = 1
    
    # запускаем inference на кастомном датасете
    saved_dir = Path("saved")
    model_files = list(saved_dir.glob("**/model_best.pth"))
    
    model_path = model_files[0] if model_files else None
    
    if model_files:
        print(f"\nИспользуем модель: {model_path}")
        
        # команда для запуска inference
        cmd = [
            "python3",
            "inference.py",
            "datasets=custom_dataset",
            f"inferencer.from_pretrained={model_path}",
            "inferencer.save_path=custom_dataset_results",
            f"dataloader.batch_size={batch_size}" 
        ]
        
        print(f"\nЗапуск команды:")
        # print(f"   {' '.join(cmd)}")
        
        # запускаем inference
        result = subprocess.run(cmd, capture_output=True, text=True)
      
        # вывод результатов
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print("Ошибки:")
            print(result.stderr)
        
        print("_________________")
        
        if result.returncode == 0:
            print("\nInference выполнен успешно!")
            
            # ищем сохраненные результаты
            results_dir = Path("data/saved/custom_dataset_results/test")
            if results_dir.exists():
                print(f"\nРезультаты сохранены в: {results_dir}")
                
                # показываем примеры файлов с предсказаниями
                prediction_files = list(results_dir.glob("*.txt"))
                if prediction_files:
                    print(f"\nНайдено файлов с предсказаниями: {len(prediction_files)}")
                    print(f"\nПримеры предсказаний (первые 3):")
                    for pred_file in prediction_files[:3]:
                        with open(pred_file, 'r', encoding='utf-8') as f:
                            pred_text = f.read().strip()
                        print(f"  {pred_file.name}: {pred_text[:50]}...")
        else:
            print(f"\nОшибка при выполнении inference (код: {result.returncode})")
else:
    print("Датасет не найден!")
    print("Сначала загрузите датасет (см. Шаг с загрузкой данных)")

### 4.3. Подсчет метрик WER/CER

Если у вас есть ground truth транскрипции, вы можете использовать скрипт `calc_metrics.py` для подсчета метрик WER и CER.

In [ ]:
print("ПОДСЧЕТ МЕТРИК WER/CER")

# пути к предсказаниям и ground truth транскрипциям
predictions_dir = Path("data/saved/custom_dataset_results/test")

# загружаем информацию о датасете из предыдущего шага
custom_dataset_dir = Path("data/custom_dataset")
info_file = custom_dataset_dir / "dataset_info.json"

if info_file.exists():
    with open(info_file, 'r') as f:
        dataset_info = json.load(f)
    transcription_dir_str = dataset_info.get("transcription_dir")
    if transcription_dir_str:
        ground_truth_dir = Path(transcription_dir_str)
    else:
        ground_truth_dir = None
else:
    # ищем папку transcriptions
    ground_truth_dir = custom_dataset_dir / "transcriptions"

if predictions_dir.exists() and ground_truth_dir and ground_truth_dir.exists():
    print(f"Предсказания: {predictions_dir}")
    print(f"Ground truth: {ground_truth_dir}")
    
    # команда для calc_metrics.py
    cmd = [
        "python3",
        "calc_metrics.py",
        str(predictions_dir),
        str(ground_truth_dir),
        "--show-samples", "5"
    ]
    
    print(f"\nЗапуск команды:")
    print(f"   {' '.join(cmd)}")
    print("_____________")
    
    # запуск calc_metrics
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # вывод результатов
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("Ошибки:")
        print(result.stderr)
    
    print("____________")
    
    if result.returncode == 0:
        print("\nМетрики успешно вычислены!")
    else:
        print(f"\nОшибка при вычислении метрик (код: {result.returncode})")
else:
    print("Не найдены предсказания или ground truth транскрипции")
    if not predictions_dir.exists():
        print(f"Предсказания не найдены: {predictions_dir}")
        print("Сначала запустите inference")

    if not ground_truth_dir or not ground_truth_dir.exists():
        print(f"Ground truth транскрипции не найдены")
        if ground_truth_dir:
            print(f"Ожидалось: {ground_truth_dir}")
        print("Убедитесь, что в датасете есть папка с .txt файлами транскрипций")

## 5. Демонстрация ASR-аугментаций

В этом разделе демонстрируются реализованные ASR-аугментации:

1. **SpecAugment** - frequency и time masking на спектрограммах (2 внутри)
2. **Speed Perturbation** - изменение скорости аудио
3. **Simple Noise** - добавление шума

Для каждой аугментации показаны:
- Аудио/спектрограммы до и после применения
- Визуализация эффекта аугментации


In [ ]:
print("ДЕМОНСТРАЦИЯ ASR-АУГМЕНТАЦИЙ")

# путь к проекту
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# загружаем пример аудио (используем первый файл из сохраненного датасета или создаем синтетическое)
try:
    # поиск аудио в сохраненном датасете
    custom_dataset_dir = Path("data/custom_dataset")
    audio_files = list(custom_dataset_dir.glob("**/*.wav")) + \
                  list(custom_dataset_dir.glob("**/*.flac"))
    
    if audio_files:
        audio_path = audio_files[0]
        print(f"Используем аудио: {audio_path}")
        waveform, sample_rate = torchaudio.load(str(audio_path))
        if waveform.shape[0] > 1:
            waveform = waveform[0:1]  # первый канал
    else:
        print("Аудио файлы не найдены")
except:
    print('не получилось сделать torch.load() по аудио из датасета')
    # создаем синтетическое аудио для демонстрации
    print("Создаем синтетическое аудио для демонстрации")
    sample_rate = 16000
    duration = 2.0  # сек
    t = torch.linspace(0, duration, int(sample_rate * duration))

    # смесь частот
    waveform = torch.sin(2 * np.pi * 440 * t) * 0.5 + \
               torch.sin(2 * np.pi * 880 * t) * 0.3 + \
               torch.randn(len(t)) * 0.1
    waveform = waveform.unsqueeze(0)  # [1, time]
    
print(f"Загружено аудио: shape={waveform.shape}, sample_rate={sample_rate}")

# Инициализируем MelSpectrogram для создания спектрограмм
mel_spectrogram = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate,
    n_mels=128,
    n_fft=512,
    hop_length=256
)

# создаем базовую спектрограмму
original_spec = mel_spectrogram(waveform)  # [1, n_mels, time]
print(f"Создана спектрограмма: shape={original_spec.shape}")


def plot_audio_and_spectrogram(waveform, spectrogram, title, sample_rate=16000):
    """Визуализация аудио и спектрограммы"""
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Аудио waveform
    if waveform.dim() > 1:
        audio_data = waveform[0].numpy()
    else:
        audio_data = waveform.numpy()
    time_axis = np.arange(len(audio_data)) / sample_rate
    
    axes[0].plot(time_axis, audio_data)
    axes[0].set_title(f"{title} - Audio Waveform")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True)
    
    # Спектрограмма
    if spectrogram.dim() > 2:
        spec_data = spectrogram[0].numpy()
    else:
        spec_data = spectrogram.numpy()
    
    # в log scale 
    spec_data_db = 20 * np.log10(spec_data + 1e-10)
    
    im = axes[1].imshow(spec_data_db, aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title(f"{title} - Spectrogram (Mel)")
    axes[1].set_xlabel("Time frames")
    axes[1].set_ylabel("Mel frequency bins")
    plt.colorbar(im, ax=axes[1], label="Magnitude (dB)")
    
    plt.tight_layout()
    return fig

fig = plot_audio_and_spectrogram(waveform, original_spec, "Original")
plt.show()

In [ ]:
# SpecAugment

print("АУГМЕНТАЦИЯ 1: SPECAUGMENT")

from src.transforms.spec_augs.spec_augment import SpecAugment

spec_aug = SpecAugment(
    freq_mask_param=27,
    num_freq_masks=2,
    time_mask_ratio=0.05,
    num_time_masks=10
)
spec_aug.train()  # вкл режим обучения

# применяем аугментацию
augmented_spec = spec_aug(original_spec.clone())
print(f"SpecAugment применен")

# визуал
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# исходная спектрограмма
spec_orig = original_spec[0].numpy()
spec_orig_db = 20 * np.log10(spec_orig + 1e-10)
im1 = axes[0].imshow(spec_orig_db, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title("До SpecAugment")
axes[0].set_xlabel("Time frames")
axes[0].set_ylabel("Mel frequency bins")
plt.colorbar(im1, ax=axes[0], label="Magnitude (dB)")

# аугментированная спектрограмма
spec_aug = augmented_spec[0].numpy()
spec_aug_db = 20 * np.log10(spec_aug + 1e-10)
im2 = axes[1].imshow(spec_aug_db, aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title("После SpecAugment (frequency + time masking)")
axes[1].set_xlabel("Time frames")
axes[1].set_ylabel("Mel frequency bins")
plt.colorbar(im2, ax=axes[1], label="Magnitude (dB)")

plt.tight_layout()
plt.show()

In [ ]:
# Perturbation

print("АУГМЕНТАЦИЯ 2: SPEED PERTURBATION")

from src.transforms.wav_augs.speed_perturbation import SpeedPerturbation

speed_aug = SpeedPerturbation(
    speeds=[0.9, 1.0, 1.1],
    sample_rate=sample_rate,
    p=1.0
)
speed_aug.train()

# применяем для разных скоростей
speeds_to_test = [0.9, 1.1]
fig, axes = plt.subplots(len(speeds_to_test) + 1, 2, figsize=(14, 4 * (len(speeds_to_test) + 1)))

# исходное аудио
audio_orig = waveform[0].numpy() if waveform.dim() > 1 else waveform.numpy()
time_orig = np.arange(len(audio_orig)) / sample_rate
axes[0, 0].plot(time_orig, audio_orig)
axes[0, 0].set_title(f"Исходное аудио (speed=1.0)")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(True)

spec_orig = mel_spectrogram(waveform)
spec_orig_plot = 20 * np.log10(spec_orig[0].numpy() + 1e-10)
im = axes[0, 1].imshow(spec_orig_plot, aspect='auto', origin='lower', cmap='viridis')
axes[0, 1].set_title("Спектрограмма исходного аудио")
axes[0, 1].set_xlabel("Time frames")
axes[0, 1].set_ylabel("Mel frequency bins")
plt.colorbar(im, ax=axes[0, 1], label="Magnitude (dB)")

for idx, speed in enumerate(speeds_to_test):
    # Применяем speed perturbation
    speed_aug_fixed = SpeedPerturbation(
        speeds=[speed],
        sample_rate=sample_rate,
        p=1.0
    )
    speed_aug_fixed.train()
    
    # применяем
    aug_waveform = speed_aug_fixed(waveform.clone())
    aug_spec = mel_spectrogram(aug_waveform)
    
    # визуал
    audio_aug = aug_waveform[0].numpy() if aug_waveform.dim() > 1 else aug_waveform.numpy()
    time_aug = np.arange(len(audio_aug)) / sample_rate
    axes[idx + 1, 0].plot(time_aug, audio_aug)
    axes[idx + 1, 0].set_title(f"После Speed Perturbation (speed={speed})")
    axes[idx + 1, 0].set_xlabel("Time (s)")
    axes[idx + 1, 0].set_ylabel("Amplitude")
    axes[idx + 1, 0].grid(True)
    
    spec_aug_plot = 20 * np.log10(aug_spec[0].numpy() + 1e-10)
    im = axes[idx + 1, 1].imshow(spec_aug_plot, aspect='auto', origin='lower', cmap='viridis')
    axes[idx + 1, 1].set_title(f"Спектрограмма (speed={speed})")
    axes[idx + 1, 1].set_xlabel("Time frames")
    axes[idx + 1, 1].set_ylabel("Mel frequency bins")
    plt.colorbar(im, ax=axes[idx + 1, 1], label="Magnitude (dB)")
    
    print(f"Speed={speed}: длина изменена с {len(audio_orig)} до {len(audio_aug)} samples")

plt.tight_layout()
plt.show()


In [ ]:
# Simple Noise
print("АУГМЕНТАЦИЯ 3: SIMPLE NOISE")

from src.transforms.wav_augs.simple_noise import SimpleNoise

# разные уровни SNR
snr_levels = [15, 5, 0]  # dB
fig, axes = plt.subplots(len(snr_levels) + 1, 2, figsize=(14, 4 * (len(snr_levels) + 1)))

# Исходное аудио
audio_orig = waveform[0].numpy() if waveform.dim() > 1 else waveform.numpy()
time_orig = np.arange(len(audio_orig)) / sample_rate
axes[0, 0].plot(time_orig, audio_orig)
axes[0, 0].set_title("Исходное аудио (без шума)")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(True)

spec_orig = mel_spectrogram(waveform)
spec_orig_plot = 20 * np.log10(spec_orig[0].numpy() + 1e-10)
im = axes[0, 1].imshow(spec_orig_plot, aspect='auto', origin='lower', cmap='viridis')
axes[0, 1].set_title("Спектрограмма исходного аудио")
axes[0, 1].set_xlabel("Time frames")
axes[0, 1].set_ylabel("Mel frequency bins")
plt.colorbar(im, ax=axes[0, 1], label="Magnitude (dB)")

for idx, snr_max in enumerate(snr_levels):
    # cоздаем noise augmentation с фиксированным snr
    noise_aug = SimpleNoise(
        noise_type="gaussian",
        snr_range=(snr_max, snr_max),
        p=1.0
    )
    noise_aug.train()
    
    # применяем
    aug_waveform = noise_aug(waveform.clone())
    aug_spec = mel_spectrogram(aug_waveform)
    
    # визуал
    audio_aug = aug_waveform[0].numpy() if aug_waveform.dim() > 1 else aug_waveform.numpy()
    time_aug = np.arange(len(audio_aug)) / sample_rate
    axes[idx + 1, 0].plot(time_aug, audio_aug)
    axes[idx + 1, 0].set_title(f"После добавления шума (SNR={snr_max} dB)")
    axes[idx + 1, 0].set_xlabel("Time (s)")
    axes[idx + 1, 0].set_ylabel("Amplitude")
    axes[idx + 1, 0].grid(True)
    
    spec_aug_plot = 20 * np.log10(aug_spec[0].numpy() + 1e-10)
    im = axes[idx + 1, 1].imshow(spec_aug_plot, aspect='auto', origin='lower', cmap='viridis')
    axes[idx + 1, 1].set_title(f"Спектрограмма (SNR={snr_max} dB)")
    axes[idx + 1, 1].set_xlabel("Time frames")
    axes[idx + 1, 1].set_ylabel("Mel frequency bins")
    plt.colorbar(im, ax=axes[idx + 1, 1], label="Magnitude (dB)")
    
    # вычисляем реальный SNR
    signal_power = (audio_orig ** 2).mean()
    noise = audio_aug - audio_orig
    noise_power = (noise ** 2).mean()
    if noise_power > 0:
        actual_snr = 10 * np.log10(signal_power / noise_power)
        print(f"Запрошенный SNR={snr_max} dB, реальный SNR≈{actual_snr:.2f} dB")

plt.tight_layout()
plt.show()